# 04 · Từ tensor PyTorch đến PINN đầu tiên

**Mục tiêu:** sau bài này, bạn tự viết được một PINN tối thiểu cho dầm Euler–Bernoulli chuẩn hóa, giải thích từng hạng loss, kiểm tra đạo hàm tự động và đánh giá nghiệm trên lưới độc lập. Notebook **tự chứa mã**, không tải script từ GitHub khi chạy trên Colab. CPU là đủ.

**Quy trình:** tensor → autograd → mạng → residual → loss → huấn luyện → kiểm chứng → một thí nghiệm cải thiện. Hãy đọc chú thích trước khi bấm **Runtime → Run all**.

## 0. Môi trường và tính tái lập

Trong Colab, chọn **Runtime → Change runtime type → CPU**. PyTorch thường đã có sẵn; chỉ cài thêm nếu ô dưới báo thiếu gói. Mỗi người có runtime riêng, nên ghi phiên bản và seed trong báo cáo.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 7
torch.manual_seed(SEED)
torch.set_num_threads(1)
DTYPE = torch.float64
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("Thiết bị:", "CPU", "| seed:", SEED, "| dtype:", DTYPE)

## 1. Tensor trước khi học mạng

Tensor giống mảng NumPy nhưng có thể tham gia **đồ thị tính toán** để PyTorch tính gradient. Với mô hình nhận một tọa độ $\xi$ và trả một chuyển vị $u$, ta dùng hình dạng `(N, 1)`: $N$ hàng là $N$ điểm, một cột là một biến. `*` nhân từng phần tử; `@` là nhân ma trận. Cố định `float64` cho bài toán cần đạo hàm bậc bốn.

In [ ]:
x_np = np.array([[0.0], [0.5], [1.0]])
x = torch.tensor(x_np, dtype=DTYPE)
print("shape", x.shape, "dtype", x.dtype, "device", x.device)
print("x**2 =", (x**2).reshape(-1).tolist())
print("x.T @ x =", (x.T @ x).item())
assert x.shape == (3, 1)

**Dừng và dự đoán:** `x*x` và `x.T @ x` có cùng hình dạng không? Chạy ô trên để đối chiếu. Khi chuyển sang mạng, lỗi shape `(N,)` so với `(N,1)` là lỗi rất thường gặp.

## 2. Autograd theo tọa độ

Muốn lấy đạo hàm theo $\xi$, phải tạo tensor đầu vào với `requires_grad=True`. Ví dụ $p(\xi)=\xi^4-2\xi^3+\xi$ có $p^{(4)}=24$. `torch.autograd.grad` trả gradient mà **không tự cập nhật trọng số mạng**. `create_graph=True` giữ đồ thị của gradient để còn đạo hàm tiếp hoặc lan truyền loss qua residual.

In [ ]:
xi = torch.tensor([[0.0], [0.25], [0.5], [0.75], [1.0]],
                  dtype=DTYPE, requires_grad=True)
p = xi**4 - 2*xi**3 + xi
dp = torch.autograd.grad(p, xi, grad_outputs=torch.ones_like(p),
                         create_graph=True)[0]
print("p'(0.25) từ autograd:", dp[1].item())
print("p'(0.25) tính tay:", (4*0.25**3 - 6*0.25**2 + 1))
assert torch.allclose(dp, 4*xi**3 - 6*xi**2 + 1)

Vì mỗi đầu ra ở hàng $i$ chỉ phụ thuộc vào tọa độ ở **cùng hàng**, `grad_outputs=ones_like(...)` cho đạo hàm theo từng điểm. Cách này cần xem lại nếu mô hình ghép các hàng với nhau (ví dụ dùng thống kê chung của batch). Viết hàm để lấy đạo hàm bậc bất kỳ:

In [ ]:
def derivative(y, x, order):
    # Mỗi hàng đầu ra chỉ phụ thuộc hàng đầu vào tương ứng.
    result = y
    for _ in range(order):
        result = torch.autograd.grad(result, x,
                                     grad_outputs=torch.ones_like(result),
                                     create_graph=True)[0]
    return result

for order in (1, 2, 3, 4):
    print(f"đạo hàm bậc {order} tại 0.25:", derivative(p, xi, order)[1].item())
assert torch.allclose(derivative(p, xi, 4), torch.full_like(xi, 24.0), atol=1e-12)

**Hai gradient khác nhau:** `derivative(u, xi, 4)` lấy đạo hàm của nghiệm theo **tọa độ** để tạo residual. `loss.backward()` lấy đạo hàm của loss theo **tham số mạng** để optimizer thay đổi tham số. Tọa độ là dữ liệu đầu vào, không phải tham số cần học.

**Tự thử:** bỏ `requires_grad=True` ở `xi` trong một ô bản sao. Ghi thông báo lỗi và giải thích. Sau đó khôi phục ô gốc, tránh làm hỏng các ô phía dưới.

## 3. Bài toán dầm chuẩn hóa và nghiệm tham khảo

Với $EI$ và tải đều $q_0$ không đổi, đặt $\xi=x/L$, $u(\xi)=EIw(L\xi)/(q_0L^4)$. Dầm tựa đơn thỏa

$$u^{(4)}(\xi)=1\quad(0<\xi<1),\qquad u(0)=u(1)=u''(0)=u''(1)=0.$$

Nghiệm đúng $u_*(\xi)=(\xi^4-2\xi^3+\xi)/24$ chỉ dùng **sau huấn luyện** để kiểm chứng. Mômen chuẩn hóa là $\bar M=-u''$. Hãy kiểm tra nghiệm bằng đạo hàm tự động:

In [ ]:
def exact_u(x):
    return (x**4 - 2*x**3 + x) / 24

check_x = torch.linspace(0, 1, 11, dtype=DTYPE).reshape(-1, 1).requires_grad_(True)
u_star = exact_u(check_x)
assert torch.allclose(derivative(u_star, check_x, 4), torch.ones_like(check_x), atol=1e-12)
assert torch.allclose(exact_u(torch.tensor([[0.], [1.]], dtype=DTYPE)),
                      torch.zeros((2, 1), dtype=DTYPE), atol=1e-12)
print("Kiểm tra PDE và chuyển vị biên: đạt")

## 4. Mạng xấp xỉ $u_\theta$

Mạng `1 → 32 → 32 → 1` nhận một $\xi$ và trả một $u_\theta(\xi)$. Mỗi `Linear` tính phép affine, `Tanh` tạo phi tuyến trơn. Residual dạng mạnh dùng đạo hàm bậc bốn nên `ReLU` không phù hợp làm baseline: mạng ReLU theo một biến là tuyến tính từng đoạn, đạo hàm bậc bốn bằng 0 hầu khắp nơi.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(1, 32), nn.Tanh(),
                         nn.Linear(32, 32), nn.Tanh(),
                         nn.Linear(32, 1)).to(dtype=DTYPE)

model = make_model()
sample = torch.tensor([[0.2], [0.8]], dtype=DTYPE)
print(model)
print("input shape", tuple(sample.shape), "output shape", tuple(model(sample).shape))
print("số tham số học được:", sum(p.numel() for p in model.parameters()))
assert model(sample).shape == (2, 1)

## 5. Điểm collocation, điều kiện biên và hàm mất mát

Chọn $N_c$ trung điểm trong $(0,1)$ để đo residual $r_\theta(\xi)=u_\theta^{(4)}(\xi)-1$. Ở hai đầu đo cả chuyển vị và $u_\theta''$:

$$\mathcal L_{\rm PDE}=\frac1{N_c}\sum_i r_\theta(\xi_i)^2,\quad
\mathcal L_{\rm BC}=u_\theta(0)^2+u_\theta(1)^2+u_\theta''(0)^2+u_\theta''(1)^2.$$

$$\mathcal L=\mathcal L_{\rm PDE}+\lambda_{\rm BC}\mathcal L_{\rm BC}.$$

Điểm biên cần `requires_grad=True` vì phải tính $u_\theta''$ ở đó. `loss_pde` lấy trung bình theo số điểm; `loss_bc` cộng bốn sai lệch, nên khi đổi $N_c$ hai hạng không tự đổi thang giống nhau.

In [ ]:
N_C = 32
LAMBDA_BC = 10.0
x_c = ((torch.arange(N_C, dtype=DTYPE) + 0.5) / N_C).reshape(-1, 1).requires_grad_(True)
x_b = torch.tensor([[0.0], [1.0]], dtype=DTYPE, requires_grad=True)
print("collocation", tuple(x_c.shape), "boundary", tuple(x_b.shape))

def loss_terms(net, interior, boundary, boundary_weight):
    u_c = net(interior)
    residual = derivative(u_c, interior, 4) - 1.0
    loss_pde = residual.square().mean()
    u_b = net(boundary)
    u_b_xx = derivative(u_b, boundary, 2)
    loss_bc = (u_b.square() + u_b_xx.square()).sum()
    return loss_pde, loss_bc, loss_pde + boundary_weight * loss_bc

lp, lb, total = loss_terms(model, x_c, x_b, LAMBDA_BC)
print(f"Trước huấn luyện: PDE={lp.item():.3e}, BC={lb.item():.3e}, total={total.item():.3e}")

## 6. Huấn luyện: mỗi dòng thay đổi điều gì?

`zero_grad()` xóa gradient tham số của bước trước; `loss.backward()` tính gradient theo tham số mạng; `optimizer.step()` cập nhật tham số. `create_graph=True` trong hàm `derivative` cho phép gradient của **loss PDE** đi ngược qua đạo hàm bậc bốn đến trọng số mạng. Ta ghi riêng hai hạng loss để thấy hạng nào đang chi phối.

In [ ]:
STEPS = 300
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
history = {"step": [], "pde": [], "bc": [], "total": []}
start = time.perf_counter()
for step in range(1, STEPS + 1):
    optimizer.zero_grad()
    loss_pde, loss_bc, loss = loss_terms(model, x_c, x_b, LAMBDA_BC)
    loss.backward()
    optimizer.step()
    history["step"].append(step)
    history["pde"].append(loss_pde.item())
    history["bc"].append(loss_bc.item())
    history["total"].append(loss.item())
    # Tọa độ là lá của đồ thị; không dùng gradient của chúng để cập nhật mạng.
    x_c.grad = None
    x_b.grad = None
elapsed = time.perf_counter() - start
print(f"{STEPS} bước trong {elapsed:.1f} s; loss {history['total'][0]:.3e} → {history['total'][-1]:.3e}")
assert np.isfinite(history["total"]).all()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for key in ("pde", "bc", "total"):
    ax.semilogy(history["step"], history[key], label=key)
ax.set(xlabel="Bước Adam", ylabel="Loss", title="Theo dõi từng hạng loss")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

**Không kết luận từ loss train một mình.** Loss đo đúng các hạng đã định nghĩa trên 32 điểm và hai đầu. Nghiệm và mômen cần được đo trên một lưới dày **khác** điểm collocation. Nếu loss dao động hoặc không giảm, kiểm tra dtype, shape, `requires_grad`, thang loss và learning rate trước khi tăng kích thước mạng.

## 7. Verify trên lưới độc lập

Đánh giá chuyển vị, mômen, residual và điều kiện biên. Chuẩn sai số tương đối dùng tích phân hình thang trên 401 điểm: $E_u=\|u_\theta-u_*\|_{L^2}/\|u_*\|_{L^2}$ và tương tự cho $\bar M$.

In [ ]:
x_eval = torch.linspace(0, 1, 401, dtype=DTYPE).reshape(-1, 1).requires_grad_(True)
u_pred = model(x_eval)
M_pred = -derivative(u_pred, x_eval, 2)
residual_eval = derivative(u_pred, x_eval, 4) - 1.0
u_ref = exact_u(x_eval)
M_ref = x_eval * (1 - x_eval) / 2

def relative_l2(pred, ref, x):
    return torch.sqrt(torch.trapz((pred[:, 0]-ref[:, 0]).square(), x[:, 0]) /
                      torch.trapz(ref[:, 0].square(), x[:, 0])).item()

E_u = relative_l2(u_pred, u_ref, x_eval)
E_M = relative_l2(M_pred, M_ref, x_eval)
u_end = model(x_b)
M_end = -derivative(u_end, x_b, 2)
print(f"E_u={E_u:.3e}, E_M={E_M:.3e}")
print(f"max |residual| trên lưới độc lập = {residual_eval.abs().max().item():.3e}")
print("u(0),u(1):", u_end.detach().reshape(-1).tolist())
print("M(0),M(1):", M_end.detach().reshape(-1).tolist())
assert np.isfinite([E_u, E_M]).all()

In [ ]:
xx = x_eval.detach().reshape(-1).tolist()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(xx, u_ref.detach().reshape(-1).tolist(), label="exact")
axes[0].plot(xx, u_pred.detach().reshape(-1).tolist(), "--", label="PINN")
axes[0].set_title("Chuyển vị u")
axes[1].plot(xx, M_ref.detach().reshape(-1).tolist(), label="exact")
axes[1].plot(xx, M_pred.detach().reshape(-1).tolist(), "--", label="PINN")
axes[1].set_title("Mômen chuẩn hóa")
axes[2].plot(xx, residual_eval.detach().reshape(-1).tolist())
axes[2].set_title("Residual trên lưới độc lập")
for ax in axes:
    ax.set_xlabel("ξ"); ax.grid(True, alpha=.3); ax.legend() if ax is not axes[2] else None
fig.tight_layout(); plt.show()

## 8. Một thí nghiệm cải thiện có kiểm soát

Chọn **một** thay đổi: `N_C = 64`, hoặc `LAMBDA_BC = 100`, hoặc tăng `STEPS`. Chạy lại từ ô tạo mô hình với **cùng seed**; ghi một hàng cho baseline và một hàng cho biến thể: số điểm, trọng số biên, số bước, thời gian, loss PDE, loss BC, $E_u$, $E_M$ và sai lệch biên. Sau đó chạy thêm ít nhất ba seed để xem mức dao động. Không dùng nghiệm $u_*$ trong loss huấn luyện cơ bản.

**Câu hỏi báo cáo:**

1. Vì sao `create_graph=True` cần cho đạo hàm bậc bốn và bước `backward()`?
2. Vì sao mạng ReLU thông thường không hợp với residual dạng mạnh này?
3. Biến thể có cải thiện **sai số độc lập** và sai lệch biên không, hay chỉ giảm loss train? Chi phí tăng bao nhiêu?
4. Hãy chỉ một vùng trên đồ thị nơi sai số chuyển vị và mômen khác nhau.

**Sản phẩm:** notebook chạy lại từ đầu, một bảng so sánh, hai hình (loss và nghiệm/mômen), và đoạn kết luận ngắn nêu giới hạn của một bài toán dầm 1D. Có thể nộp báo cáo Word hoặc LaTeX.